# modulos/ledger — o ledger do pipeline (`pipeline_manifest`, docs/arquitetura)

Módulo carregado pelo hub `environment/env_functions`. O livro-razão do pipeline: UMA linha por etapa × `ano_edicao` em que cada etapa declara o **contrato** do que produziu — quantas linhas, quantos indicadores com dados, quanto de recusa preservou, a partir de qual arquivo (SHA-256).

## Blocos deste módulo

* **Família do contrato** — `gravar_linha_manifest`, `contrato_da_edicao`, `check_igual` + `resumo_validacoes`
* **Medição do shape da edição** — `colunas_com_dados`, `celulas_recusa`
* **Verificação do arquivo da landing** — `verificar_arquivo_landing`

## Papéis (desenho, seção 1.1)

O ledger é primariamente **contrato** (validado nas fronteiras por `check_igual`), secundariamente **auditoria** (cada atualização da linha é uma versão no `DESCRIBE HISTORY` da própria tabela) e deliberadamente **não é evidência** — isso continua com as `dq_*` do bronze_quality_profiling.

O MAP `detalhes` guarda contexto, diagnóstico e relatório: não entra nos asserts por padrão, mas localiza o desvio quando um check reprova — e sobe a coluna contratual quando passa a ser sistematicamente validado (seção 5.5).

`manifest_tbl` vem do `env_variables`; a tabela nasce na primeira gravação.

## Família do contrato

Uma linha por etapa × `ano_edicao` em que cada etapa declara o CONTRATO do que produziu; as etapas posteriores validam contra ela na fronteira (nenhum número hardcoded no consumidor).

* `gravar_linha_manifest` — grava/atualiza a linha (MERGE na chave; a tabela nasce na primeira gravação; `detalhes` como MAP)
* `contrato_da_edicao` — retorna a linha como dict; aborta com instrução quando ausente e quando as tabelas físicas declaradas não confirmam o contrato (DROP manual = contrato desmentido)
* `check_igual` + `resumo_validacoes` — `check_igual` REGISTRA um check PASS/FAIL (obtido vs. esperado do contrato) e não interrompe; `resumo_validacoes` fecha o lote e interrompe o run com todas as falhas de uma vez (fail-fast em bloco, padrão do Step 9 da gold)

`manifest_tbl` vem do `env_variables`.

In [0]:
import io
import contextlib
from delta.tables import DeltaTable
from pyspark.sql import functions as F

SCHEMA_MANIFEST = """
  etapa            STRING,
  ano_edicao       INT,
  tabela_destino   STRING,
  sha256_xlsx      STRING,
  linhas           BIGINT,
  indicadores      INT,
  recusa_total     BIGINT,
  detalhes         MAP<STRING, STRING>,
  criado_em        TIMESTAMP
"""


def gravar_linha_manifest(etapa, ano_edicao, tabela_destino="", sha256_xlsx=None,
                          linhas=None, indicadores=None, recusa_total=None, detalhes=None):
    """Grava/atualiza a linha etapa×ano_edicao do ledger (MERGE na chave).

    É o estado atual, não o histórico: re-rodar a etapa atualiza a linha, nunca
    duplica (o histórico completo das escritas continua no DESCRIBE HISTORY de
    cada tabela — e do próprio ledger). Chamada por cada etapa ao concluir sua
    gravação: landing (ingest), bronze, profiling, silver e gold.
    """
    if not spark.catalog.tableExists(manifest_tbl):
        spark.sql(f"CREATE TABLE IF NOT EXISTS {manifest_tbl} ({SCHEMA_MANIFEST}) USING DELTA")
        print(f"Criada: {manifest_tbl}")

    df_linha = spark.createDataFrame(
        [(etapa, int(ano_edicao), tabela_destino or None, sha256_xlsx,
          int(linhas) if linhas is not None else None,
          int(indicadores) if indicadores is not None else None,
          int(recusa_total) if recusa_total is not None else None)],
        schema="etapa string, ano_edicao int, tabela_destino string, sha256_xlsx string, "
               "linhas bigint, indicadores int, recusa_total bigint",
    )
    if detalhes:
        pares = [F.lit(x) for k, v in sorted(detalhes.items()) for x in (k, str(v))]
        mapa = F.create_map(*pares)
    else:
        mapa = F.lit(None).cast("map<string,string>")
    df_linha = (df_linha
                .withColumn("detalhes", mapa)
                .withColumn("criado_em", F.current_timestamp()))

    # MERGE pela API do Delta: userMetadata só chega ao histórico via conf de sessão
    # (indisponível em compute serverless — o _set_user_metadata avisa uma vez e segue)
    _set_user_metadata(f"Ledger: {etapa} - edicao {ano_edicao}")

    # O commit do MERGE emite ~2 linhas em branco no stdout (ruído da Delta em
    # serverless) — capturamos e reimprimimos apenas as linhas com conteúdo
    _stdout_merge = io.StringIO()
    with contextlib.redirect_stdout(_stdout_merge):
        (DeltaTable.forName(spark, manifest_tbl).alias("target")
            .merge(df_linha.alias("source"),
                   "target.etapa = source.etapa AND target.ano_edicao = source.ano_edicao")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())
    for _linha in _stdout_merge.getvalue().splitlines():
        if _linha.strip():
            print(_linha)

    print(f"Ledger atualizado: etapa '{etapa}', edição {ano_edicao} -> {manifest_tbl}")


def contrato_da_edicao(etapa, ano_edicao):
    """Retorna a linha do ledger (o contrato) da etapa×edição, como dict.

    Aborta com instrução clara quando a linha não existe (padrão da
    _exigir_dq_da_edicao da gold) e também quando as tabelas físicas que a
    linha declara já não confirmam o contrato (DROP manual): o ledger declara,
    a guard confere (seção 5.4 do desenho).
    """
    if not spark.catalog.tableExists(manifest_tbl):
        raise RuntimeError(
            f"Ledger ausente: {manifest_tbl} não existe. As etapas do pipeline gravam "
            f"suas linhas ao executar (comece pelo pipeline/ingest_to_landing)."
        )
    linhas = (spark.table(manifest_tbl)
              .filter((F.col("etapa") == etapa) & (F.col("ano_edicao") == int(ano_edicao)))
              .collect())
    if not linhas:
        raise RuntimeError(
            f"Contrato ausente no ledger: não há linha da etapa '{etapa}' para a edição "
            f"{ano_edicao} em {manifest_tbl}. Rode a etapa '{etapa}' desta edição primeiro "
            f"(ela declara seu contrato ao concluir a gravação)."
        )
    contrato = linhas[0].asDict()

    # A existência real é confirmada na tabela física, não no ledger (DROP manual)
    for tabela in [t.strip() for t in (contrato.get("tabela_destino") or "").split(";") if t.strip()]:
        if tabela.startswith("/"):
            continue  # caminho de volume (linha 'landing'), não é tabela — nada a conferir
        if not spark.catalog.tableExists(tabela):
            raise RuntimeError(
                f"Contrato desmentido: o ledger declara '{etapa}/{ano_edicao}', mas a tabela "
                f"{tabela} não existe (apagada manualmente?). Rode novamente a etapa "
                f"'{etapa}' da edição {ano_edicao}."
            )
        if "ano_edicao" in spark.table(tabela).columns:
            n = spark.table(tabela).filter(F.col("ano_edicao") == int(ano_edicao)).count()
            if n == 0:
                raise RuntimeError(
                    f"Contrato desmentido: {tabela} existe, mas não tem linhas da edição "
                    f"{ano_edicao}. Rode novamente a etapa '{etapa}' desta edição."
                )
    return contrato


_VALIDACOES = []  # acumula os check_igual até o resumo_validacoes


def check_igual(obtido, esperado, nome_check, contexto=""):
    """Registra um check de igualdade contra o contrato — não interrompe o run.

    'obtido' é o valor medido agora; 'esperado' vem da linha da etapa no ledger
    (contrato_da_edicao). Um FAIL acumula e todas as reprovações interrompem
    juntas no resumo_validacoes (padrão do Step 9 do silver_to_gold: checks
    PASS/FAIL). 'contexto' entra na mensagem de falha e serve para citar os
    valores de 'detalhes' da linha do ledger — o diagnóstico que localiza o
    desvio sem uma segunda consulta (seção 5.5).
    """
    ok = obtido == esperado
    _VALIDACOES.append({"check": nome_check, "esperado": esperado, "obtido": obtido})
    print(f"[{'PASS' if ok else 'FAIL'}] {nome_check}: esperado {esperado}, obtido {obtido}")
    if not ok and contexto:
        print(f"    contexto: {contexto}")
    return ok


def resumo_validacoes():
    """Fecha o lote de check_igual: resume PASS/FAIL e, se algo reprovou,
    interrompe com TODAS as falhas listadas de uma vez (fail-fast em bloco)."""
    falhas = [v for v in _VALIDACOES if v["obtido"] != v["esperado"]]
    print(f"\nValidação contra o contrato: {len(_VALIDACOES) - len(falhas)}/{len(_VALIDACOES)} checks PASS")
    if falhas:
        lista = "\n".join(
            f"  - {f['check']}: esperado {f['esperado']}, obtido {f['obtido']}" for f in falhas
        )
        raise AssertionError(f"{len(falhas)} check(s) reprovado(s) contra o contrato do ledger:\n{lista}")
    _VALIDACOES.clear()

## Medição do shape da edição

Utilidades usadas ao declarar e validar contratos do ledger:

* `colunas_com_dados(df, cols)` — no schema união (multi-edição), colunas que só existem em outra edição chegam 100% nulas para a corrente: o campo `indicadores` do manifest declara o shape **com dados** da edição
* `celulas_recusa(df, cols)` — total de células 'Recusa': nasce na bronze e segue intacta até a `fact_resposta` da gold (invariante da cadeia, campo `recusa_total`)

Antes viviam inline em 4 pontos (landing_to_bronze célula do ledger, bronze_to_silver no fail-fast e no Step 8, silver_to_gold Step 1).

In [0]:
from functools import reduce


def colunas_com_dados(df, cols):
    """Colunas de `cols` com ao menos um valor não-nulo em `df` (a edição corrente)."""
    m = df.agg(
        *[F.max(F.col(c).isNotNull().cast("int")).alias(c) for c in cols]
    ).collect()[0].asDict()
    return [c for c in cols if m[c] == 1]


def celulas_recusa(df, cols):
    """Total de células 'Recusa' nas colunas de indicadores indicadas."""
    return df.agg(
        reduce(
            lambda a, b: a + b,
            [F.sum(F.when(F.col(c) == "Recusa", 1).otherwise(0)) for c in cols],
        ).alias("total")
    ).collect()[0]["total"]

## Verificação do arquivo da landing (linha 'landing' do ledger)

O que é específico da landing (o artefato é ARQUIVO, não tabela): SHA-256 recalculado em TODO run, aviso de divergência contra o último declarado (a linha nunca é atualizada em silêncio) e `baixado_em` honesto (now se este run baixou; mtime do arquivo se já existia).

`verificar_arquivo_landing` **não grava** no ledger — retorna `(sha256_xlsx, detalhes)`; quem declara é o notebook, chamando `gravar_linha_manifest` como toda etapa faz (desenho, §5.6).

In [0]:
import datetime
import hashlib
import os


def verificar_arquivo_landing(ano_edicao, url_origem, file_path, baixou):
    """Verifica o arquivo da edição e monta os valores da linha 'landing'.

    baixou: retorno de download_file (True se este run baixou, False se já existia).
    Retorna (sha256_xlsx, detalhes) prontos para o gravar_linha_manifest.
    """
    file_name = url_origem.split("/")[-1]
    with open(file_path, "rb") as f:
        file_bytes = f.read()
    file_sha256 = hashlib.sha256(file_bytes).hexdigest()

    print(f"Arquivo : {file_name}")
    print(f"Volume  : {file_path}")
    print(f"Tamanho : {len(file_bytes):,} bytes")
    print(f"SHA-256 : {file_sha256}\n")

    # Último contrato declarado — para avisar divergência (seção 5.6)
    sha_anterior = None
    if spark.catalog.tableExists(manifest_tbl):
        linhas_ledger = (
            spark.table(manifest_tbl)
            .filter((F.col("etapa") == "landing") & (F.col("ano_edicao") == int(ano_edicao)))
            .select("sha256_xlsx")
            .collect()
        )
        if linhas_ledger and linhas_ledger[0]["sha256_xlsx"]:
            sha_anterior = linhas_ledger[0]["sha256_xlsx"]

    # 'baixado_em': quando o arquivo chegou ao volume — agora se este run baixou;
    # mtime do arquivo se já existia (nunca um 'agora' fabricado)
    momento_download = (
        datetime.datetime.now() if baixou
        else datetime.datetime.fromtimestamp(os.path.getmtime(file_path))
    )
    detalhes = {
        "arquivo": file_name,
        "tamanho_mb": f"{len(file_bytes) / (1024 * 1024):.1f}",
        "url_origem": url_origem,
        "baixado_em": momento_download.strftime("%Y-%m-%d %H:%M"),
        "ja_existia": str(not baixou).lower(),
    }
    if sha_anterior and sha_anterior != file_sha256:
        print(f"AVISO: SHA-256 da edição {ano_edicao} divergiu do último declarado no ledger "
              f"— o insumo da landing mudou (exclusão + re-download, origem alterada "
              f"ou arquivo substituído).")
        print(f"    anterior: {sha_anterior}")
        print(f"    novo    : {file_sha256}\n")
        detalhes["sha_anterior_divergente"] = sha_anterior

    return file_sha256, detalhes